## Job Hunter Assistant with Persistent Memory

Multi-agent MCPs that help hunt jobs and store notes in a db


In [12]:
import os
import json
from pathlib import Path
from datetime import datetime

from dotenv import load_dotenv
from openai import AsyncOpenAI
from IPython.display import display, Markdown
from contextlib import AsyncExitStack

from agents import Agent, Runner, Tool, trace, OpenAIChatCompletionsModel, set_tracing_disabled
from agents.mcp import MCPServerStdio

load_dotenv(override=True)

True

In [13]:

job_server_path = os.path.abspath("job_server.py")
job_search_params = {"command": "python", "args": [job_server_path]}

fetch_params = {"command": "uvx", "args": ["mcp-server-fetch"]}

sandbox_path = os.path.abspath("sandbox")
os.makedirs(sandbox_path, exist_ok=True)
files_params = {
    "command": "npx",
    "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path],
}


MCP_TIMEOUT = 200

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_MODEL = "openai/gpt-4o-mini"

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY is required in .env")

set_tracing_disabled(True)

openrouter_client = AsyncOpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=OPENROUTER_API_KEY,
)

llm_model = OpenAIChatCompletionsModel(
    model=OPENROUTER_MODEL,
    openai_client=openrouter_client,
)

In [14]:
async def get_scout_tool(mcp_servers) -> Tool:
    instructions = f"""You help users hunt for jobs from plain-language goals (e.g. mobile dev 2026, AI engineering, backend).

There is no keyword web search. If the user message includes http(s) URLs, use the fetch MCP tool on those URLs and fold what you learn into notes.
Otherwise use your knowledge to produce useful, honest notes: skills in demand, role types, how to search, what to put in a portfolio — do not invent specific live job postings.

Persist with the job MCP tools:
- Pick one short SYMBOL for this focus (e.g. MOBILE, AIENG, BACKEND). save_company(symbol, name, sector): name/sector should reflect the user’s goal.
- save_job_note(symbol, content): rich note (landscape, skills, example places to look, stack hints).
- save_job_plan(symbol, plan): concrete next steps (apply weekly, projects to build, networking, follow-up when they paste URLs).

Current date: {datetime.now().strftime("%Y-%m-%d")}
"""
    scout = Agent(
        name="Scout",
        instructions=instructions,
        model=llm_model,
        mcp_servers=mcp_servers,
    )
    return scout.as_tool(
        tool_name="Scout",
        tool_description="From a natural-language job goal, save company row + notes + plan; fetch any URLs the user provided.",
    )


async def get_organizer_tool(mcp_servers) -> Tool:
    instructions = """You organize the job search.
Read jobs://all and jobs://company/{SYMBOL} resources.
Optionally write a markdown summary to the sandbox (e.g. summary.md).
"""
    organizer = Agent(
        name="Organizer",
        instructions=instructions,
        model=llm_model,
        mcp_servers=mcp_servers,
    )
    return organizer.as_tool(
        tool_name="Organizer",
        tool_description="Read jobs:// resources and write files to sandbox.",
    )

In [15]:
async def run_job_hunter(query: str):
    async with AsyncExitStack() as stack:
        job_server = await stack.enter_async_context(
            MCPServerStdio(job_search_params, client_session_timeout_seconds=MCP_TIMEOUT)
        )
        fetch_server = await stack.enter_async_context(
            MCPServerStdio(fetch_params, client_session_timeout_seconds=MCP_TIMEOUT)
        )
        files_server = await stack.enter_async_context(
            MCPServerStdio(files_params, client_session_timeout_seconds=MCP_TIMEOUT)
        )

        scout_mcps = [job_server, fetch_server]
        scout_tool = await get_scout_tool(scout_mcps)
        organizer_tool = await get_organizer_tool([job_server, files_server])

        main_instructions = """The user asks in normal language (e.g. find mobile jobs for 2026, AI engineering, backend developer).

1. Call Scout with their goal — do not ask them to phrase save_company or MCP steps.
2. Optionally call Organizer to read jobs://all or jobs://company/{SYMBOL} and write sandbox summary.md.
3. Answer in short, friendly prose: what you saved and suggested next steps.
"""
        coordinator = Agent(
            name="JobCoordinator",
            instructions=main_instructions,
            model=llm_model,
            tools=[scout_tool, organizer_tool],
        )

        with trace("job_hunter"):
            result = await Runner.run(coordinator, query, max_turns=30)
            display(Markdown(result.final_output))

### Example queries (plain text)

Run one at a time. Symbols (e.g. `MOBILE`, `AIENG`) are chosen by the agent — inspect with **`jobs://all`** below.

In [16]:
await run_job_hunter("Find jobs for me in mobile development in 2026.")
# await run_job_hunter("I'm looking for AI engineering roles.")
# await run_job_hunter("Help me with backend developer opportunities.")

I've gathered insights and a plan for pursuing mobile development jobs in 2026. Here's a quick summary:

### Mobile Development Overview
- **Skills Needed**: Swift, Kotlin, UI/UX design, and experience with frameworks like React Native and Flutter.
- **Role Types**: Positions such as Mobile Application Developer and UI/UX Designer.
- **Where to Search**: Job boards (LinkedIn, Glassdoor), company sites, and tech meetups.

### Job Search Plan
1. Update your resume and LinkedIn with relevant experiences.
2. Build a portfolio of mobile app projects.
3. Apply weekly to relevant positions.
4. Join online mobile development communities for networking.
5. Attend tech meetups for industry insights.

You're all set to dive into the mobile development job market! If you have specific companies or resources you'd like to check out, let me know!

### Read everything saved (`jobs://all`)

In [17]:
async with MCPServerStdio(job_search_params, client_session_timeout_seconds=MCP_TIMEOUT) as server:
    result = await server.session.read_resource("jobs://all")
    for content in result.contents:
        print(content.text)

{
  "total_companies": 1,
  "companies": [
    {
      "symbol": "MOBILE",
      "name": "Mobile Development",
      "sector": "Technology",
      "updated_at": "2026-03-27T15:57:53.458514"
    }
  ]
}


### List all companies

In [18]:
async with MCPServerStdio(job_search_params, client_session_timeout_seconds=MCP_TIMEOUT) as server:
    result = await server.session.read_resource("jobs://all")
    for content in result.contents:
        data = json.loads(content.text)
        print(f"Companies: {data['total_companies']}")
        for c in data["companies"]:
            print(f"  - {c['symbol']}: {c['name']} ({c.get('sector', '')})")

Companies: 1
  - MOBILE: Mobile Development (Technology)
